In [0]:
def check(name, query):
    result = spark.sql(query).first()[0]
    assert result == 0, f"FAILED {name}: {result} bad rows"
    print(f"passed: {name}")

check("silver orders unique", "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM workspace.silver.orders")
check("no negative quantities", "SELECT COUNT(*) FROM workspace.silver.orders WHERE quantity <= 0")
check("no orphan customers", """
  SELECT COUNT(*) FROM workspace.silver.orders o
  LEFT JOIN workspace.silver.customers c ON o.customer_id = c.customer_id
  WHERE c.customer_id IS NULL""")
check("status values valid", """
  SELECT COUNT(*) FROM workspace.silver.orders
  WHERE status NOT IN ('completed','shipped','cancelled','returned')""")

In [0]:
check("silver events unique", "SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM workspace.silver.events")
check("event types valid", """
  SELECT COUNT(*) FROM workspace.silver.events
  WHERE event_type NOT IN ('page_view','add_to_cart','purchase')""")
check("no null customers in events", "SELECT COUNT(*) FROM workspace.silver.events WHERE customer_id IS NULL")

In [0]:
check("product_conversion covers all products", """
  SELECT COUNT(*) FROM workspace.silver.products p
  LEFT JOIN workspace.gold.product_conversion c ON p.product_id = c.product_id
  WHERE c.product_id IS NULL""")
check("conversion rates in range", """
  SELECT COUNT(*) FROM workspace.gold.product_conversion
  WHERE view_to_purchase < 0 OR view_to_purchase > 1""")